# Прављење апликација за генерисање слика (OpenAI)

Постоји више од једноставне генерације текста уз LLM моделе. Такође можете генерисати слике из текстуалних описа. Слике као модалитет су корисне у Медицинској технологији, архитектури, туризму, развоју игара, маркетингу и још много тога. У овој лекцији радимо са данашњим **GPT Image** моделима на OpenAI платформи.

## Циљеви учења

До краја ове лекције ћете моћи да:

- Објасните шта је генерисање слика и где је корисно.
- Разумете породицу модела `gpt-image` и како се разликују од старих DALL·E модела.
- Направите апликацију за генерисање слика и уређујете слике.

## Шта је генерисање слика?

Модели за генерисање слика стварају слике из текстуалних упутстава. Модерни модели као што је `gpt-image` уче везу између текста и слика током тренинга, затим итеративно претварају насумични шум у слику која одговара вашем упутству.

Две добро познате породице модела за слике су:

- **`gpt-image` (OpenAI)** — актуелна генерација коришћена у овој лекцији. Подржава генерисање слике из текста и уређивање слика (инпаинтинг са маском).
- **Midjourney** — популаран модел треће стране са сопственом услугом и Discord-овим радним процесом.

> Старији OpenAI модели слика — **DALL·E 2** и **DALL·E 3** — су застарели. DALL·E 3 више није доступан за нове примене, а функције као што је `create_variation` постојале су само у DALL·E 2. За нове апликације користите `gpt-image` моделе.

> **Важно:** `gpt-image` модели враћају генерисану слику као **base64** (`b64_json`), а не као URL. Ваш код декодира base64 стринг у бајтове и чува је — нема URL адресу слике за преузимање.


## Прављење ваше прве апликације за генерисање слика

Шта је све потребно за прављење апликације за генерисање слика? Потребне су вам следеће библиотеке:

- **python-dotenv**, препоручује се да користите ову библиотеку да бисте чували своје тајне у *.env* фајлу одвојено од кода.
- **openai**, ову библиотеку ћете користити за интеракцију са OpenAI API-јем.
- **pillow**, за рад са сликама у Пајтону.


1. Направите фајл *.env* са следећим садржајем:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Прикупите горе наведене библиотеке у фајл који се зове *requirements.txt* на следећи начин:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Затим, направите виртуелно окружење и инсталирајте библиотеке:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> За Виндоус, користите следеће команде за креирање и активирање вашег виртуелног окружења:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Додајте следећи код у датотеку која се зове *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Хајде да објаснимо овај код:

- Прво, увозимо потребне библиотеке, укључујући OpenAI библиотеку, dotenv библиотеку, base64 модул и Pillow библиотеку.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Након тога, креирамо клијента који чита API кључ из вашег ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Следеће, генеришемо слику:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` модели враћају слику као **base64** стринг у `data[0].b64_json`. Декодујемо га у бајтове и пишемо у датотеку — нема URL за преузимање.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- На крају, отварамо слику и користимо стандардни прегледач слика да је прикажемо:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Више детаља о генерисању слике

Погледајмо параметре `images.generate`:

- **model**, је модел слике који се користи (на пример `gpt-image-2`).
- **prompt**, је текстуални упит који се користи за генерисање слике. Овде је "Zeka на коњу, носи лизалицу, на магловитој ливади где расту нарциси".
- **size**, је величина генерисане слике (на пример `1024x1024`, `1536x1024`, `1024x1536`, или `"auto"`).
- **n**, је број генерисаних слика. Овде генеришемо једну.

> Модели слика не узимају `temperature` параметар — то је контрола за текстуалну генерацију. За више варијетета, позовите API поново; да бисте смањили варијетет, учините ваш упит прецизнијим.

## Додатне могућности генерисања слика

Видећете како се слика генерише уз неколико редова Python-а. `gpt-image` модели такође могу **уредити** постојећу слику. Преласком сликом, опционалном **маском** (која означава подручје за промену) и упитом, можете изменити део слике — на пример, додати шешир нашем зеки.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Основна слика садржи само зека; коначна слика има шешир на зеку.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
